# 03 · MNIST test mẫu

Notebook này chỉ đọc `mnist_weights.json`, không train lại model. Dùng notebook này để kiểm tra một mẫu test, xem dự đoán của cả bốn model và quan sát probability.

Chạy sau `mnist_dataset.ipynb` và `mnist_train.ipynb`. Có thể đổi `sample_index` để kiểm tra mẫu khác.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'gk' / 'web' / 'backend').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from gk.web.backend.digits_predict import _forward_model

DATASET_DIR = PROJECT_ROOT / 'gk' / 'web' / 'backend' / 'artifacts' / 'mnist_dataset'
DATASET_PATH = DATASET_DIR / 'mnist_normalized.npz'
WEIGHTS_PATH = DATASET_DIR / 'mnist_weights.json'
data = np.load(DATASET_PATH)
weights = json.loads(WEIGHTS_PATH.read_text(encoding='utf-8'))
X_test_images = data['X_test_pixels'].astype(float)
y_test = data['y_test'].astype(int)
X_test_pixels = X_test_images.reshape(len(X_test_images), -1)
mean = np.asarray(weights['preprocessing']['mean'], dtype=float)
std = np.asarray(weights['preprocessing']['std'], dtype=float)
models = weights['models']
assert X_test_images.shape == (10000, 8, 8)
assert len(models) == 4
print('Loaded weights:', WEIGHTS_PATH)
print('Models:', [model['name'] for model in models])

## Chọn mẫu cần kiểm tra

Đổi số này rồi chạy lại các cell phía dưới. Đây là index trong test set MNIST, không phải index train.

In [ ]:
sample_index = 7
assert 0 <= sample_index < len(X_test_pixels)
pixels = X_test_pixels[sample_index]
normalized = (pixels - mean) / std
predictions = [_forward_model(model, normalized) for model in models]

print('sample index:', sample_index)
print('true label  :', int(y_test[sample_index]))
for result in predictions:
    print(f"{result['name']:32} → {result['predicted_class']} ({result['confidence']:.2%})")

repeat = [_forward_model(model, normalized)['probabilities'] for model in models]
assert all(np.allclose(first['probabilities'], second) for first, second in zip(predictions, repeat))
print('deterministic forward pass: OK')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(pixels.reshape(8, 8), cmap='gray_r', vmin=0, vmax=16)
axes[0].set_title(f'test sample {sample_index} · true label {y_test[sample_index]}')
axes[0].set_xticks(range(8))
axes[0].set_yticks(range(8))
axes[0].grid(color='white', linewidth=0.5, alpha=0.4)

width = 0.2
digits = np.arange(10)
for offset, result in enumerate(predictions):
    axes[1].bar(digits + offset * width, result['probabilities'], width, label=result['name'])
axes[1].set_xticks(digits + width * 1.5, digits)
axes[1].set_ylim(0, 1.05)
axes[1].set_xlabel('digit')
axes[1].set_ylabel('probability')
axes[1].set_title('Four models')
axes[1].legend(fontsize=8)
plt.tight_layout()

In [ ]:
selected_model_id = 'compact-tuned'
selected = next(result for result in predictions if result['id'] == selected_model_id)
print('Selected model:', selected['name'])
for layer in selected['layers']:
    values = np.asarray(layer['h'])
    print(f"{layer['name']:16} activation={layer['activation']:8} units={len(values):2} max={values.max():.4f}")

hidden_layers = selected['layers'][:-1]
if hidden_layers:
    fig, axes = plt.subplots(1, len(hidden_layers), squeeze=False, figsize=(5 * len(hidden_layers), 3))
    for axis, layer in zip(axes[0], hidden_layers):
        axis.bar(np.arange(len(layer['h'])), layer['h'])
        axis.set_title(f"{layer['name']} · {layer['activation']}")
        axis.set_xlabel('neuron')
    plt.tight_layout()